# SAS821S Capstone T08 - Mining Remote Operations and Industrial-IoT Intrusion Analytics
**Group:** 222093471 Sophia Kalume (Member A, data & modelling) · 215103815 Lasarus Shithindi (Member B, security engineering & intelligence)

This notebook walks through the implemented pipeline and reproduces every number used in the
report and the presentation. It reads the outputs written by `python src/run_all.py`.

**Data status:** ToN_IoT (UNSW Canberra) research captures are real; the remote-access log,
maintenance tickets, asset inventory and user directory are **synthetic**, generated by
documented rules with a fixed seed. The four abnormal access patterns were injected on
purpose and are used as ground truth. Nothing here describes a real organisation or a real
incident.

In [ ]:
import json, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
T, F, P = ROOT/'outputs'/'tables', ROOT/'outputs'/'figures', ROOT/'data'/'processed'
def tbl(n): return pd.read_csv(T/n)
def js(n): return json.loads((T/n).read_text())
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

## Step 1-2 · Data inventory and requirement coverage

In [ ]:
inv = tbl('data_inventory.csv')
print('total records:', f"{inv['rows'].sum():,}")
display(inv[['dataset','rows','columns','data_status','project_role','time_from','time_to','labelled']])
display(tbl('minimum_data_check.csv'))

## Step 3 · Cleaning log (what was changed and why)

In [ ]:
clog = tbl('cleaning_log.csv')
print(len(clog), 'logged actions')
display(clog.head(15))
display(tbl('data_quality.csv').head(10))

## Step 4 · Baselines and exploratory findings

In [ ]:
display(tbl('eda_findings.csv'))
display(tbl('baseline_access_role.csv').round(2))
from IPython.display import Image, display as show
for f in ['fig01_telemetry_volume_over_time.png','fig08_login_hour_by_role.png']:
    show(Image(filename=str(F/f)))

## Step 5 · Supervised detection
Models are compared on cross-validated F1 on the training split; the test split is used once.
Exact duplicate rows are removed **before** the split, otherwise the same flow can appear in
both train and test.

In [ ]:
display(tbl('supervised_network_cv.csv').round(4))
display(tbl('supervised_network_test.csv').round(4))
display(tbl('supervised_modbus_test.csv').round(4))
show(Image(filename=str(F/'fig09_confusion_network.png')))
show(Image(filename=str(F/'fig11_confusion_network_multiclass.png')))

## Step 6 · Anomaly detection, and an important negative result
Fully unsupervised outlier detection performs **below chance** on this capture because
attack traffic is the majority class. Learning a clean baseline first fixes it.

In [ ]:
a = js('anomaly_summary.json')
print('fully unsupervised ROC-AUC :', round(a['network']['roc_auc_unsupervised'],3))
print('clean-baseline ROC-AUC     :', round(a['network']['roc_auc_clean_baseline'],3))
print(a['network']['interpretation'])
display(tbl('anomaly_access_validation.csv').round(3))
display(tbl('anomaly_access_flagged_user_days.csv').head(10))

## Step 7 · Access analytics (UEBA rules with evidence)

In [ ]:
display(tbl('ueba_rule_summary.csv').round(3))
display(tbl('ueba_alerts.csv').head(12))

## Step 8 · Investigation
Check the `linkage_basis` column: rows marked CONTEXT ONLY come from a different source
environment and share only the calendar window.

In [ ]:
inc = js('incident_summary.json'); print(json.dumps(inc, indent=1)[:900])
display(tbl('incident_timeline.csv').head(15))
display(tbl('incident_response_plan.csv'))

## Step 9 · Intelligence products

In [ ]:
display(tbl('intel_requirements.csv'))
display(tbl('intel_attack_mapping.csv'))
display(tbl('intel_operational_watchlist.csv').head(10))
print((ROOT/'outputs'/'intelligence_executive_brief.md').read_text()[:1800])

## Step 10 · Control simulation

In [ ]:
display(tbl('simulation_summary.csv').round(3))
display(tbl('simulation_sensitivity.csv').round(3).head(15))
show(Image(filename=str(F/'fig20_simulation_scenarios.png')))
print(json.dumps(js('simulation_summary.json')['limitations'], indent=1))

## Step 11 · Text mining

In [ ]:
n = js('nlp_summary.json')
print('corpus:', n['corpus_sizes'])
print('classification macro-F1:', round(n['classification']['macro_f1'],3))
print('CAVEAT:', n['classification']['caveat'])
print('indicator extraction:', {k: round(v,3) for k,v in n['indicator_extraction'].items() if isinstance(v,float)})
display(tbl('nlp_attack_topics.csv'))
display(tbl('nlp_extracted_indicators.csv').head(8))

## Step 12-13 · Predictive risk and adversarial testing

In [ ]:
display(tbl('risk_band_summary.csv').round(3))
display(tbl('risk_user_day_scores.csv').head(10))
display(tbl('risk_forecast_metrics.csv').round(3))
display(tbl('adversarial_results.csv').round(4))
show(Image(filename=str(F/'fig25_adversarial_tests.png')))

## Step 15 · Architecture and Step 20 · Compliance

In [ ]:
display(tbl('architecture_stages.csv'))
display(tbl('compliance_checklist.csv')[['code','requirement','status','files_present','remaining_work']])
show(Image(filename=str(F/'fig26_architecture.png')))